<a href="https://colab.research.google.com/github/JoyNgaru/NSE-2025-dataset-analysis/blob/main/NSE_Data_Warehouse_and_Risk_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NSE Stocks Data Warehouse & Risk/Return Intelligence
**Stack:** Google Colab + DuckDB + scikit-learn + Looker Studio

This notebook builds a small analytics stack on top of the Nairobi Securities
Exchange (NSE) daily price data (2023–2025):

1. **Ingest** the 3 raw CSVs into DuckDB (bronze layer)
2. **Clean** them into a typed, deduplicated table (silver layer)
3. **Model** a star-schema warehouse: `dim_company`, `dim_date`, `fact_daily_price` (gold layer)
4. **Engineer features**: daily returns, annualised volatility, drawdown, liquidity
5. **Cluster** companies by risk/return profile with K-Means (unsupervised ML)
6. **Score** companies for risk-averse vs. risk-taking investors
7. **Export** clean summary tables ready to plug into Google Sheets → Looker Studio

> ⚠️ **Not investment advice.** This is a historical, descriptive analysis of
> 2023–2025 price data. Past performance on ~3 years of data is not a reliable
> predictor of future returns, and NSE has many thinly-traded stocks where
> price moves are exaggerated by low liquidity. Treat outputs as a
> screening/exploration tool, not a recommendation.


## 0. Setup

In [5]:
# DuckDB isn't preinstalled on Colab — install it once per session
!pip install duckdb --quiet

import duckdb
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', 50)
print("duckdb:", duckdb.__version__, "| sklearn:", sklearn.__version__)


duckdb: 1.3.2 | sklearn: 1.6.1


## 1. Upload the raw data

Upload the three Kaggle CSVs (`NSE_data_all_stocks_2023.csv`,
`NSE_data_all_stocks_2024.csv`, `NSE_data_all_stocks_2025.csv`).


In [8]:
from google.colab import files

uploaded = files.upload()   # select all 3 CSVs from the picker
csv_files = {
    2023: "1NSE_data_all_stocks_2023.csv",
    2024: "2NSE_data_all_stocks_2024.csv",
    2025: "3NSE_data_all_stocks_2025.csv",
}
for yr, fname in csv_files.items():
    assert fname in uploaded, f"Missing {fname} — please re-upload"
print("All 3 files present.")
print(uploaded.keys())



Saving 1NSE_data_all_stocks_2023.csv to 1NSE_data_all_stocks_2023.csv
Saving 2NSE_data_all_stocks_2024.csv to 2NSE_data_all_stocks_2024.csv
Saving 3NSE_data_all_stocks_2025.csv to 3NSE_data_all_stocks_2025.csv
All 3 files present.
dict_keys(['1NSE_data_all_stocks_2023.csv', '2NSE_data_all_stocks_2024.csv', '3NSE_data_all_stocks_2025.csv'])


## 2. Bronze layer — load raw CSVs into DuckDB

We create/connect to a single DuckDB file `nse_warehouse.duckdb` — this file
*is* your data warehouse. Everything below (staging, dims, facts, marts) lives
inside it as tables/views, so you can reopen it later without re-running the
whole pipeline.


In [9]:
con = duckdb.connect("nse_warehouse.duckdb")

# The 2025 file has a UTF-8 BOM on its header; duckdb's csv reader handles
# this fine. We tag each row with its source file/year for traceability.
con.execute("DROP TABLE IF EXISTS raw_prices")
con.execute("""
    CREATE TABLE raw_prices AS
    SELECT *, 2023 AS source_year FROM read_csv_auto(?, header=True, all_varchar=True)
    UNION ALL BY NAME
    SELECT *, 2024 AS source_year FROM read_csv_auto(?, header=True, all_varchar=True)
    UNION ALL BY NAME
    SELECT *, 2025 AS source_year FROM read_csv_auto(?, header=True, all_varchar=True)
""", [csv_files[2023], csv_files[2024], csv_files[2025]])

print(con.execute("SELECT source_year, COUNT(*) FROM raw_prices GROUP BY 1 ORDER BY 1").df())


   source_year  count_star()
0         2023         17274
1         2024         18119
2         2025         15114


## 3. Silver layer — clean & type the data

Cleaning issues found in the raw data:
- Numbers use `-` for "no value" (not zero) and commas as thousand separators (`"1,900.00"`)
- `Change%` has a trailing `%`
- `Adjusted Price` is `-` for every single row (0% populated) → dropped
- `Date` is formatted like `3-Jan-23` → needs explicit parsing
- Codes starting with `^` (e.g. `^NASI`, `^N20I`) are **market indices**, not
  companies — kept separately as benchmarks, excluded from company rankings
- A few codes are ETFs/REITs/preference shares/rights (`GLD`, `SMWF`, `LAPR`,
  `KPLC-P4`, `HFCK-R`, …) — flagged by instrument type rather than dropped


In [10]:
con.execute("DROP TABLE IF EXISTS clean_prices")
con.execute("""
    CREATE TABLE clean_prices AS
    SELECT
        strptime(Date, '%d-%b-%y')::DATE                       AS trade_date,
        Code                                                     AS code,
        Name                                                     AS name,
        TRY_CAST(NULLIF(REPLACE("12m Low", ',', ''), '-') AS DOUBLE)  AS low_52w,
        TRY_CAST(NULLIF(REPLACE("12m High", ',', ''), '-') AS DOUBLE) AS high_52w,
        TRY_CAST(NULLIF(REPLACE("Day Low", ',', ''), '-') AS DOUBLE)  AS day_low,
        TRY_CAST(NULLIF(REPLACE("Day High", ',', ''), '-') AS DOUBLE) AS day_high,
        TRY_CAST(NULLIF(REPLACE("Day Price", ',', ''), '-') AS DOUBLE) AS day_price,
        TRY_CAST(NULLIF(REPLACE(Previous, ',', ''), '-') AS DOUBLE)    AS previous_price,
        TRY_CAST(NULLIF(REPLACE(Change, ',', ''), '-') AS DOUBLE)      AS change_abs,
        TRY_CAST(NULLIF(REPLACE(REPLACE("Change%", ',', ''), '%',''), '-') AS DOUBLE) AS change_pct,
        TRY_CAST(NULLIF(REPLACE(Volume, ',', ''), '-') AS DOUBLE)      AS volume,
        source_year
    FROM raw_prices
    WHERE Code IS NOT NULL
""")

# de-duplicate exact (date, code) repeats if any slipped in
con.execute("""
    CREATE OR REPLACE TABLE clean_prices AS
    SELECT * FROM clean_prices
    QUALIFY ROW_NUMBER() OVER (PARTITION BY trade_date, code ORDER BY source_year DESC) = 1
""")

print(con.execute("SELECT COUNT(*), COUNT(DISTINCT code), MIN(trade_date), MAX(trade_date) FROM clean_prices").df())


   count_star()  count(DISTINCT code) min(trade_date) max(trade_date)
0         50507                    78      2023-01-03      2025-10-31


## 4. Gold layer — star-schema warehouse

- **`dim_company`** — one row per ticker, with an `instrument_type` flag
  (Ordinary Share / ETF / REIT / Preference Share / Rights / Index)
- **`dim_date`** — calendar attributes for easy slicing in Looker Studio
- **`fact_daily_price`** — one row per (company, date), the grain of the warehouse


In [11]:
con.execute("DROP TABLE IF EXISTS dim_company")
con.execute("""
    CREATE TABLE dim_company AS
    SELECT DISTINCT
        code,
        FIRST(name) OVER (PARTITION BY code) AS name,
        CASE
            WHEN code LIKE '^%'      THEN 'Index'
            WHEN code IN ('GLD','SMWF') THEN 'ETF'
            WHEN code = 'LAPR'       THEN 'REIT'
            WHEN code LIKE '%-P%'    THEN 'Preference Share'
            WHEN code LIKE '%-R'     THEN 'Rights'
            ELSE 'Ordinary Share'
        END AS instrument_type
    FROM clean_prices
""")

con.execute("DROP TABLE IF EXISTS dim_date")
con.execute("""
    CREATE TABLE dim_date AS
    SELECT DISTINCT
        trade_date                        AS date,
        YEAR(trade_date)                  AS year,
        QUARTER(trade_date)               AS quarter,
        MONTH(trade_date)                 AS month,
        strftime(trade_date, '%B')        AS month_name,
        DAYOFWEEK(trade_date)             AS day_of_week,
        strftime(trade_date, '%A')        AS day_name
    FROM clean_prices
""")

con.execute("DROP TABLE IF EXISTS fact_daily_price")
con.execute("""
    CREATE TABLE fact_daily_price AS
    SELECT
        trade_date AS date, code, low_52w, high_52w, day_low, day_high,
        day_price, previous_price, change_abs, change_pct, volume,
        (volume IS NOT NULL AND volume > 0) AS is_trading_day
    FROM clean_prices
""")

print("dim_company:", con.execute("SELECT COUNT(*) FROM dim_company").fetchone()[0])
print(con.execute("SELECT instrument_type, COUNT(*) FROM dim_company GROUP BY 1 ORDER BY 2 DESC").df())


dim_company: 78
    instrument_type  count_star()
0    Ordinary Share            64
1             Index             8
2               ETF             2
3  Preference Share             2
4            Rights             1
5              REIT             1


## 5. Feature engineering — returns, volatility, drawdown, liquidity

Two important design choices, both because ~35% of company-days have no
trades at all (price just sits at the last quote):

1. **Returns/volatility are computed only across actual trading days**
   (`is_trading_day = TRUE`), so a stale, untraded quote isn't mistaken for a
   real 0% return.
2. **CAGR/total return use the first and last available price** in the
   window (trading or not), since that's the price you'd actually
   buy/sell at.


In [12]:
con.execute("DROP TABLE IF EXISTS fact_daily_returns")
con.execute("""
    CREATE TABLE fact_daily_returns AS
    SELECT
        date, code, day_price, volume,
        day_price / LAG(day_price) OVER (PARTITION BY code ORDER BY date) - 1 AS daily_return
    FROM fact_daily_price
    WHERE is_trading_day
""")

print(con.execute("SELECT * FROM fact_daily_returns WHERE code = 'SCOM' ORDER BY date DESC LIMIT 5").df())


        date  code  day_price      volume  daily_return
0 2025-10-31  SCOM      30.25   3051822.0      0.023689
1 2025-10-30  SCOM      29.55   2509371.0      0.001695
2 2025-10-29  SCOM      29.50   8235098.0      0.000000
3 2025-10-28  SCOM      29.50  15906412.0      0.029668
4 2025-10-27  SCOM      28.65   1932055.0      0.012367


In [13]:
# Per-company, per-year metrics (for year-over-year trend charts)
con.execute("DROP TABLE IF EXISTS company_yearly_metrics")
con.execute("""
    CREATE TABLE company_yearly_metrics AS
    WITH trading AS (
        SELECT YEAR(date) AS year, code, date, day_price, volume, daily_return
        FROM fact_daily_returns
    ),
    firsts_lasts AS (
        SELECT year, code,
               FIRST(day_price ORDER BY date)  AS first_price,
               LAST(day_price ORDER BY date)   AS last_price,
               COUNT(*)                        AS trading_days,
               AVG(volume)                     AS avg_volume,
               STDDEV_SAMP(daily_return) * SQRT(252) AS ann_volatility
        FROM trading
        GROUP BY 1, 2
    )
    SELECT
        f.*,
        (last_price / first_price - 1) AS year_return
    FROM firsts_lasts f
""")
print(con.execute("SELECT * FROM company_yearly_metrics ORDER BY code, year LIMIT 8").df())


   year  code  first_price  last_price  trading_days     avg_volume  \
0  2023  ABSA        12.20       11.45           243  510164.197531   
1  2024  ABSA        11.55       18.05           249  447821.285141   
2  2025  ABSA        18.85       23.00           208  705828.096154   
3  2025  AMAC        70.00       59.75            34     194.852941   
4  2023  BAMB        31.45       35.85           235   86088.510638   
5  2024  BAMB        36.00       55.00           246  682288.699187   
6  2025  BAMB        55.00       54.00            42    7809.523810   
7  2023   BAT       459.50      407.50           233   20132.618026   

   ann_volatility  year_return  
0        0.217674    -0.061475  
1        0.223352     0.562771  
2        0.229125     0.220159  
3        1.278225    -0.146429  
4        0.455948     0.139905  
5        0.631203     0.527778  
6        0.289194    -0.018182  
7        0.234238    -0.113166  


In [14]:
# Whole-period (3-year) metrics + max drawdown, joined to dim_company
con.execute("DROP TABLE IF EXISTS company_overall_metrics")
con.execute("""
    CREATE TABLE company_overall_metrics AS
    WITH base AS (
        SELECT code, date, day_price, volume, daily_return,
               MAX(day_price) OVER (PARTITION BY code ORDER BY date
                                     ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_peak
        FROM fact_daily_returns
    ),
    dd AS (
        SELECT code, MIN(day_price / running_peak - 1) AS max_drawdown
        FROM base GROUP BY 1
    ),
    agg AS (
        SELECT
            code,
            FIRST(day_price ORDER BY date) AS first_price,
            LAST(day_price ORDER BY date)  AS last_price,
            COUNT(*)                       AS trading_days,
            AVG(volume)                    AS avg_daily_volume,
            STDDEV_SAMP(daily_return) * SQRT(252) AS ann_volatility,
            (POWER(LAST(day_price ORDER BY date) / FIRST(day_price ORDER BY date),
                   365.25 / DATEDIFF('day', MIN(date), MAX(date))) - 1) AS cagr_3yr
        FROM fact_daily_returns
        GROUP BY 1
        HAVING COUNT(*) >= 30   -- drop near-dead / newly-listed tickers with too little history
    )
    SELECT
        a.code, c.name, c.instrument_type,
        a.trading_days, a.avg_daily_volume, a.ann_volatility, a.cagr_3yr,
        (a.last_price / a.first_price - 1) AS total_return_3yr,
        dd.max_drawdown,
        a.cagr_3yr / NULLIF(a.ann_volatility, 0) AS return_per_risk
    FROM agg a
    JOIN dim_company c USING (code)
    LEFT JOIN dd USING (code)
""")
print(con.execute("SELECT COUNT(*) FROM company_overall_metrics").fetchone()[0], "companies with enough history")
con.execute("SELECT * FROM company_overall_metrics ORDER BY total_return_3yr DESC LIMIT 10").df()


60 companies with enough history


,code,name,instrument_type,trading_days,avg_daily_volume,ann_volatility,cagr_3yr,total_return_3yr,max_drawdown,return_per_risk
0,KPLC,Kenya Power and Lighting Company Plc,Ordinary Share,700,1.001953e+06,0.551242,1.128927,7.456790,-0.330153,2.047969
1,PORT,East African Portland Cement Ltd,Ordinary Share,502,3.952189e+03,0.824074,1.088243,6.975871,-0.596059,1.320565
2,SMER,Sameer Africa Plc,Ordinary Share,618,3.263041e+04,0.783880,0.998707,6.075472,-0.318996,1.274057
3,ORCH,Kenya Orchards Ltd,Ordinary Share,40,4.611500e+04,0.852181,1.584916,5.140351,-0.069767,1.859835
4,HFCK,HF Group Ltd,Ordinary Share,699,3.304735e+05,0.555255,0.539004,2.380952,-0.426752,0.970733
5,HAFR,Home Afrika Ltd,Ordinary Share,697,1.865838e+05,0.701415,0.529702,2.323529,-0.544643,0.755190
6,KEGN,Kenya Electricity Generating Company Plc,Ordinary Share,699,1.196038e+06,0.381782,0.506528,2.183230,-0.398773,1.326747
7,OCH,Olympia Capital Holdings Ltd,Ordinary Share,535,1.567710e+04,0.739621,0.410347,1.641892,-0.518657,0.554808
8,IMH,I & M Holdings Plc,Ordinary Share,700,1.463041e+05,0.516576,0.397562,1.574780,-0.388321,0.769609
9,CIC,CIC Insurance Group Ltd,Ordinary Share,700,1.818601e+05,0.427053,0.369091,1.429319,-0.258993,0.864275


## 6. Machine learning — risk/return clustering (K-Means)

We cluster **Ordinary Shares** (excluding indices/ETFs/preference shares,
which aren't comparable) on three standardised features:

- 3-year total return
- annualised volatility
- log(average daily volume) — a liquidity proxy

K-Means finds groups; we then **label the clusters after the fact** by
looking at each cluster's centroid (high/low return, high/low risk) rather
than hard-coding cluster order, since K-Means cluster numbers are arbitrary.


In [15]:
mart = con.execute("""
    SELECT * FROM company_overall_metrics
    WHERE instrument_type = 'Ordinary Share'
      AND ann_volatility IS NOT NULL
      AND avg_daily_volume > 0
""").df()

features = mart.copy()
features["log_liquidity"] = np.log1p(features["avg_daily_volume"])
X_cols = ["total_return_3yr", "ann_volatility", "log_liquidity"]
X = features[X_cols].dropna()
features = features.loc[X.index]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
features["cluster"] = kmeans.fit_predict(X_scaled)

# Label clusters by their centroid characteristics (median return/vol per cluster)
centroid_stats = features.groupby("cluster")[["total_return_3yr", "ann_volatility"]].median()
ret_rank = centroid_stats["total_return_3yr"].rank(ascending=False)
vol_rank = centroid_stats["ann_volatility"].rank(ascending=False)

def label_cluster(c):
    high_return = ret_rank[c] <= 2
    high_vol = vol_rank[c] <= 2
    if high_return and not high_vol:
        return "Steady Compounders (return-rich, calmer)"
    if high_return and high_vol:
        return "High-Growth / High-Volatility"
    if not high_return and not high_vol:
        return "Low-Volatility Laggards"
    return "Value Traps / High-Risk Underperformers"

cluster_label_map = {c: label_cluster(c) for c in centroid_stats.index}
features["risk_profile"] = features["cluster"].map(cluster_label_map)

print(centroid_stats.assign(label=centroid_stats.index.map(cluster_label_map)))
features[["code","name","total_return_3yr","ann_volatility","avg_daily_volume","risk_profile"]]\
    .sort_values("total_return_3yr", ascending=False).head(15)


         total_return_3yr  ann_volatility  \
cluster                                     
0                0.604278        0.599641   
1                0.210843        0.723478   
2                0.762145        0.329939   
3                6.525672        0.803977   

                                            label  
cluster                                            
0                         Low-Volatility Laggards  
1         Value Traps / High-Risk Underperformers  
2        Steady Compounders (return-rich, calmer)  
3                   High-Growth / High-Volatility  


,code,name,total_return_3yr,ann_volatility,avg_daily_volume,risk_profile
23,KPLC,Kenya Power and Lighting Company Plc,7.456790,0.551242,1.001953e+06,High-Growth / High-Volatility
24,PORT,East African Portland Cement Ltd,6.975871,0.824074,3.952189e+03,High-Growth / High-Volatility
15,SMER,Sameer Africa Plc,6.075472,0.783880,3.263041e+04,High-Growth / High-Volatility
40,ORCH,Kenya Orchards Ltd,5.140351,0.852181,4.611500e+04,High-Growth / High-Volatility
50,HFCK,HF Group Ltd,2.380952,0.555255,3.304735e+05,Low-Volatility Laggards
8,HAFR,Home Afrika Ltd,2.323529,0.701415,1.865838e+05,Low-Volatility Laggards
18,KEGN,Kenya Electricity Generating Company Plc,2.183230,0.381782,1.196038e+06,"Steady Compounders (return-rich, calmer)"
52,OCH,Olympia Capital Holdings Ltd,1.641892,0.739621,1.567710e+04,Value Traps / High-Risk Underperformers
2,IMH,I & M Holdings Plc,1.574780,0.516576,1.463041e+05,Low-Volatility Laggards
35,CIC,CIC Insurance Group Ltd,1.429319,0.427053,1.818601e+05,"Steady Compounders (return-rich, calmer)"


## 7. Composite scores — best pick for a risk-averse vs. a risk-taking investor

Both scores are 0–100, built from the same standardised features but weighted
differently:

- **Risk-averse score** rewards good *risk-adjusted* return (return per unit
  of volatility) and liquidity (can actually exit the position), and
  penalises deep drawdowns.
- **Risk-taker score** rewards raw upside (total return) more heavily and
  doesn't penalise volatility — a risk-taker is fine with a bumpy ride if the
  destination is higher.

These are transparent, weighted scoring formulas (not a black-box model) so
you can see and adjust exactly why a company ranks where it does.


In [16]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

f = features.copy()
f["norm_return"]      = minmax(f["total_return_3yr"])
f["norm_low_vol"]     = minmax(-f["ann_volatility"])          # lower vol = better, for risk-averse
f["norm_high_vol"]    = minmax(f["ann_volatility"])           # higher vol = more "risk-taker" fuel
f["norm_liquidity"]   = minmax(f["log_liquidity"])
f["norm_drawdown"]    = minmax(f["max_drawdown"].fillna(f["max_drawdown"].min()))  # closer to 0 (less negative) = better

f["risk_averse_score"] = 100 * (
    0.40 * f["norm_return"] +
    0.30 * f["norm_low_vol"] +
    0.15 * f["norm_liquidity"] +
    0.15 * f["norm_drawdown"]
)

f["risk_taker_score"] = 100 * (
    0.60 * f["norm_return"] +
    0.25 * f["norm_high_vol"] +
    0.15 * f["norm_liquidity"]
)

print("=== Top 5 picks for a RISK-AVERSE investor ===")
print(f.sort_values("risk_averse_score", ascending=False)[
    ["code","name","total_return_3yr","ann_volatility","risk_profile","risk_averse_score"]].head(5))

print("\n=== Top 5 picks for a RISK-TAKER ===")
print(f.sort_values("risk_taker_score", ascending=False)[
    ["code","name","total_return_3yr","ann_volatility","risk_profile","risk_taker_score"]].head(5))

company_scores = f  # keep full table for export


=== Top 5 picks for a RISK-AVERSE investor ===
    code                                      name  total_return_3yr  \
23  KPLC      Kenya Power and Lighting Company Plc          7.456790   
15  SMER                         Sameer Africa Plc          6.075472   
40  ORCH                        Kenya Orchards Ltd          5.140351   
20  ABSA                       ABSA Bank Kenya Plc          0.885246   
18  KEGN  Kenya Electricity Generating Company Plc          2.183230   

    ann_volatility                              risk_profile  \
23        0.551242             High-Growth / High-Volatility   
15        0.783880             High-Growth / High-Volatility   
40        0.852181             High-Growth / High-Volatility   
20        0.223283  Steady Compounders (return-rich, calmer)   
18        0.381782  Steady Compounders (return-rich, calmer)   

    risk_averse_score  
23          81.815719  
15          63.820086  
40          63.217069  
20          59.985129  
18          59.

## 8. Best / worst performers — overall and by year

In [17]:
print("=== Best performing companies, 2023-2025 (total return) ===")
print(company_scores.sort_values("total_return_3yr", ascending=False)
      [["code","name","total_return_3yr","ann_volatility","risk_profile"]].head(10))

print("\n=== Worst performing companies, 2023-2025 (total return) ===")
print(company_scores.sort_values("total_return_3yr")
      [["code","name","total_return_3yr","ann_volatility","risk_profile"]].head(10))


=== Best performing companies, 2023-2025 (total return) ===
    code                                      name  total_return_3yr  \
23  KPLC      Kenya Power and Lighting Company Plc          7.456790   
24  PORT          East African Portland Cement Ltd          6.975871   
15  SMER                         Sameer Africa Plc          6.075472   
40  ORCH                        Kenya Orchards Ltd          5.140351   
50  HFCK                              HF Group Ltd          2.380952   
8   HAFR                           Home Afrika Ltd          2.323529   
18  KEGN  Kenya Electricity Generating Company Plc          2.183230   
52   OCH              Olympia Capital Holdings Ltd          1.641892   
2    IMH                        I & M Holdings Plc          1.574780   
35   CIC                   CIC Insurance Group Ltd          1.429319   

    ann_volatility                              risk_profile  
23        0.551242             High-Growth / High-Volatility  
24        0.824074   

In [18]:
# Year-by-year winners/losers, straight from the warehouse
yearly = con.execute("""
    SELECT y.year, y.code, c.name, y.year_return, y.ann_volatility, y.avg_volume
    FROM company_yearly_metrics y
    JOIN dim_company c USING (code)
    WHERE c.instrument_type = 'Ordinary Share' AND y.trading_days >= 20
""").df()

for yr in sorted(yearly["year"].unique()):
    sub = yearly[yearly["year"] == yr].sort_values("year_return", ascending=False)
    print(f"\n--- {yr}: Top 5 ---")
    print(sub[["code","name","year_return"]].head(5).to_string(index=False))
    print(f"--- {yr}: Bottom 5 ---")
    print(sub[["code","name","year_return"]].tail(5).to_string(index=False))



--- 2023: Top 5 ---
code                     name  year_return
UMME                Umeme Ltd     1.139037
KAPC  Kapchorua Tea Kenya Plc     0.857451
EVRD Eveready East Africa Ltd     0.500000
 WTK Williamson Tea Kenya Plc     0.344103
EGAD              Eaagads Ltd     0.328571
--- 2023: Bottom 5 ---
code                    name  year_return
SCOM           Safaricom Plc    -0.422037
 KCB           KCB Group Plc    -0.428944
UNGA          Unga Group Ltd    -0.456452
 TCL       Trans-Century Plc    -0.474747
CGEN Car and General (K) Ltd    -0.489796

--- 2024: Top 5 ---
code                                 name  year_return
PORT     East African Portland Cement Ltd     2.825000
ORCH                   Kenya Orchards Ltd     2.589744
KPLC Kenya Power and Lighting Company Plc     2.435714
 IMH                   I & M Holdings Plc     1.077364
 KCB                        KCB Group Plc     0.895216
--- 2024: Bottom 5 ---
code                               name  year_return
SASN               

## 9. Persist results back into the warehouse
So the scored/clustered table lives in DuckDB alongside the raw facts, not
just in a pandas dataframe that disappears when the runtime resets.


In [19]:
con.execute("DROP TABLE IF EXISTS company_scores")
con.register("company_scores_df", company_scores)
con.execute("CREATE TABLE company_scores AS SELECT * FROM company_scores_df")
print(con.execute("SELECT COUNT(*) FROM company_scores").fetchone()[0], "rows written to company_scores")


58 rows written to company_scores


## 10. Export for Looker Studio

**Looker Studio cannot connect to a DuckDB file or a Colab runtime directly.**
The simplest free bridge is: export → Google Sheets → connect Looker Studio to
that Sheet. This cell writes the CSVs and (optionally) pushes them straight
into a Google Sheet in your Drive.


In [20]:
import os
os.makedirs("export", exist_ok=True)

company_scores.to_csv("export/company_scores.csv", index=False)
yearly.to_csv("export/company_yearly_metrics.csv", index=False)

# Monthly-aggregated price trend (keeps the file small enough for Sheets/Looker)
monthly_prices = con.execute("""
    SELECT code, date_trunc('month', date) AS month, AVG(day_price) AS avg_price,
           SUM(volume) AS total_volume
    FROM fact_daily_price
    GROUP BY 1, 2
""").df()
monthly_prices.to_csv("export/monthly_prices.csv", index=False)

dim_company_df = con.execute("SELECT * FROM dim_company").df()
dim_company_df.to_csv("export/dim_company.csv", index=False)

print("Files written to ./export/:")
print(os.listdir("export"))


Files written to ./export/:
['company_yearly_metrics.csv', 'company_scores.csv', 'monthly_prices.csv', 'dim_company.csv']


In [21]:
# OPTIONAL: push straight to a new Google Sheet so Looker Studio can read it live.
# Uncomment and run if you'd rather not manually download+upload the CSVs.

from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("NSE_Dashboard_Data")
for name, df in [
     ("company_scores", company_scores),
     ("company_yearly_metrics", yearly),
     ("monthly_prices", monthly_prices),
     ("dim_company", dim_company_df),
 ]:
     ws = sh.add_worksheet(title=name, rows=len(df)+10, cols=len(df.columns)+2)
     ws.update([df.columns.tolist()] + df.astype(str).values.tolist())
print("Spreadsheet created:", sh.url)


Spreadsheet created: https://docs.google.com/spreadsheets/d/1J4X8HgwkFKRDpRPCAkyU26QXEv9xNXpDFyFYQzbuItQ


In [ ]:
# Also download the whole warehouse file so you can reopen it later without
# re-running the pipeline (duckdb.connect("nse_warehouse.duckdb"))
con.close()
from google.colab import files
files.download("nse_warehouse.duckdb")


## Next step
Open **`NSE_Looker_Studio_Dashboard_Spec.md`** (delivered alongside this
notebook) for the exact pages, charts, filters and calculated fields to build
in Looker Studio using the 4 CSVs in `export/` (or the Google Sheet, if you
used the optional push cell above).
